### AUTOLOADER
### Dim User

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col
import os
import sys

prg_path = os.path.join(os.getcwd(), '..','..')

sys.path.append(prg_path)

from utils.Transform import resuable


In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimUser/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@saazurede.dfs.core.windows.net/DimUser")

obj_user = resuable()
df_user = obj_user.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(["user_id"])

df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimUser/_checkpoint")\
    .trigger(availableNow=True)\
    .option("path","abfss://silver@saazurede.dfs.core.windows.net/DimUser/data")\
    .toTable("Spotify_cate.silver.DimUser")

In [0]:
df_art = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimArtist/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@saazurede.dfs.core.windows.net/DimArtist")

obj_art = resuable()
df_art = obj_art.dropColumns(df_art, ['_rescued_data'])
df_art = df_art.dropDuplicates(["artist_id"])

df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimArtist/_checkpoint")\
    .trigger(availableNow=True)\
    .option("path","abfss://silver@saazurede.dfs.core.windows.net/DimArtist/data")\
    .toTable("Spotify_cate.silver.DimArtist")

In [0]:
df_date = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimDate/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@saazurede.dfs.core.windows.net/DimDate")

obj_date = resuable()
df_date = obj_date.dropColumns(df_date, ['_rescued_data'])
df_dare = df_date.dropDuplicates(["date_key"])


df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimDate/_checkpoint")\
    .trigger(availableNow=True)\
    .option("path","abfss://silver@saazurede.dfs.core.windows.net/DimDate/data")\
    .toTable("Spotify_cate.silver.DimDate")

In [0]:
dbutils.fs.rm("abfss://silver@saazurede.dfs.core.windows.net/DimTrack/data", True)
dbutils.fs.rm("abfss://silver@saazurede.dfs.core.windows.net/DimTrack/_checkpoint", True)
dbutils.fs.rm("abfss://silver@saazurede.dfs.core.windows.net/DimTrack/checkpoint", True)

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimTrack/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@saazurede.dfs.core.windows.net/DimTrack")

obj_track = resuable()
df_track = obj_track.dropColumns(df_track, ['_rescued_data'])
df_track = df_track.dropDuplicates(["track_id"])
df_track = df_track.withColumn("Duration_Flag", when(col("duration_sec")<150,"low")
                                               .when(col("duration_sec")<300,"medium")
                                               .otherwise("high"))\
                    .withColumn("track_name", regexp_replace(col("track_name"),"-"," "))

df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@saazurede.dfs.core.windows.net/DimTrack/_checkpoint")\
    .trigger(availableNow=True)\
    .option("mergeSchema", "true")\
    .option("path","abfss://silver@saazurede.dfs.core.windows.net/DimTrack/data")\
    .toTable("Spotify_cate.silver.DimTrack")

In [0]:
df_fctstream = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@saazurede.dfs.core.windows.net/FactStream/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@saazurede.dfs.core.windows.net/FactStream")

obj_stream = resuable()
df_fctstream = obj_stream.dropColumns(df_fctstream, ['_rescued_data'])
df_fctstream = df_fctstream.dropDuplicates(["Stream_id"])

df_fctstream.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@saazurede.dfs.core.windows.net/FactStream/_checkpoint")\
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@saazurede.dfs.core.windows.net/FactStream/data")\
    .toTable("Spotify_cate.silver.FactStream")

In [0]:
demo_df = spark.read.format("parquet")\
         .option("inferSchema", "true")\
         .option("header", "true")\
         .load("abfss://tvbronze@saazurede.dfs.core.windows.net/DimChannel")

demo_df.display()